# M0 answer-only audit v2

Revised baseline: 1,260 non-final prompts, shared Answer: format, 128-token generation cap and per-response EOS/cap logging. Preserve the earlier 32-token run. Push the v2 code first, then connect this notebook to a Kaggle GPU kernel.
Use answer_scoring.answer_matches for v2 answer counts; exact_matches is the unchanged legacy whole-response diagnostic.

In [ ]:
import os
import subprocess
import sys
import tempfile
from pathlib import Path

# This path exists only in the remote Kaggle runtime.
remote_root = Path('/kaggle/working')
assert remote_root.is_dir(), 'Connect to the Kaggle remote kernel first.'
repo_dir = Path(tempfile.mkdtemp(prefix='m0-audit-source-', dir=remote_root)) / 'repo'
subprocess.run(['git', 'clone', '--branch', 'main',
    'https://github.com/DevulapalliTharun/persistent-llm-unlearning.git', str(repo_dir)], check=True)
os.chdir(repo_dir)
print('Repository commit:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())
assert Path('configs/evaluation/m0_knowledge_audit_v2.json').is_file(), 'Push the audit files first.'

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet',
    '-r', 'requirements/m0-smoke.txt'], check=True)

In [ ]:
import json

# Plan-only verifies the candidate without loading any model.
plan_dir = Path(tempfile.mkdtemp(prefix='m0-audit-v2-plan-', dir=remote_root))
subprocess.run([sys.executable, '-m', 'src.m0_knowledge_audit',
    '--config', 'configs/evaluation/m0_knowledge_audit_v2.json',
    '--output-dir', str(plan_dir), '--plan-only'], check=True)
plan = json.loads((plan_dir / 'plan.json').read_text())
assert plan['planned_predictions'] == 1260
assert plan['knowledge_units'] == 180
print('Candidate checksums verified; ready for GPU inference.')

In [ ]:
# A new directory preserves any previous audit run.
output_dir = Path(tempfile.mkdtemp(prefix='m0-knowledge-audit-v2-', dir=remote_root))
print('Audit outputs:', output_dir, flush=True)
subprocess.run([sys.executable, '-u', '-m', 'src.m0_knowledge_audit',
    '--config', 'configs/evaluation/m0_knowledge_audit_v2.json',
    '--output-dir', str(output_dir)], check=True)

In [ ]:
summary = json.loads((output_dir / 'summary.json').read_text())
assert summary['execution_complete']
assert summary['predictions'] == 1260
assert summary['model_executed']
short_summary = {key: summary[key] for key in ('execution_complete', 'predictions', 'protocol_version', 'runtime', 'seconds', 'predictions_sha256')}
short_summary['answer_scoring'] = {key: value for key, value in summary['answer_scoring'].items() if key != 'by_partition_and_task'}
print(json.dumps(short_summary, indent=2))
print('Preserve summary.json, predictions.jsonl, matches_for_review.json, plan.json, environment.freeze.txt')
print('Saved output directory:', output_dir)
print('Review any token_limit_hits or format_violations before comparing models.')


In [ ]:
import shutil

archive = shutil.make_archive(str(output_dir) + '-results', 'zip', root_dir=str(output_dir))
print('Download from the Kaggle Output panel:', archive)


Preserve all five output files and the printed ZIP. Review raw answers, extracted answers, token-limit hits and format violations. Keep the final-test prompts and original dataset unchanged. This notebook performs inference only; v2 results remain pending until execution.